# Notebook 05 — Entrepôt multidimensionnel en étoile (DuckDB)

## Objectif

Construire l'entrepôt décisionnel gold au **format étoile** pour alimenter les outils BI.

## Schéma en étoile

```
                 dim_usine
                     │
    dim_temps  ─── fact_production ─── dim_produit
                     │
                 dim_machine
```

- **fact_production** : 1 ligne = 1 pièce produite (grain transactionnel le plus fin)
- **4 dimensions** : usine, produit, temps, machine

## Justification (étoile vs flocon vs grappe)

| Modèle | Avantages | Inconvénients | Choix MECHA |
|---|---|---|---|
| **Étoile** | Simple, performant, lisible métier | Redondance dimensionnelle | ✅ Retenu |
| **Flocon** | Moins de redondance | Requêtes plus complexes (jointures) | Non |
| **Grappe** | Groupes de faits reliés | Complexité élevée | Non |

L'**étoile** est le choix standard pour un PoC BI : lisibilité, performance, requêtage simple.

## Rattachement grille MSPR

| Compétence | Critère |
|---|---|
| C7 — Créer un entrepôt unique | Modèle multidimensionnel conçu et argumenté |
| C2 — Architecture BI | 3 couches : collecte / modélisation / restitution |


## 1. Imports

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SILVER_DIR = PROJECT_ROOT / "data" / "silver"
GOLD_DIR   = PROJECT_ROOT / "data" / "gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = GOLD_DIR / "mecha.duckdb"
if DB_PATH.exists():
    DB_PATH.unlink()

print(f"DB : {DB_PATH}")

DB : /home/romaric420/MSPR/data/gold/mecha.duckdb


## 2. Chargement silver

In [2]:
frames = [pd.read_parquet(f) for f in sorted(SILVER_DIR.glob("*.parquet"))]
df = pd.concat(frames, ignore_index=True)
print(f"Silver : {len(df)} pieces, {df['usine'].nunique()} usines")
df.head(2)

Silver : 10070 pieces, 2 usines


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF,timestamp,usine,pays
0,319,L47498,L,297.8,308.4,1512,39.596343,189,0,0,0,0,0,0,2025-10-01 06:18:00.648612845,ES-01,Espagne
1,1150,L48329,L,297.2,308.1,1446,38.351136,118,0,0,0,0,0,0,2025-10-01 06:24:32.591039184,ES-01,Espagne


## 3. Connexion à DuckDB

In [3]:
con = duckdb.connect(str(DB_PATH))
print(f"DuckDB version : {duckdb.__version__}")

DuckDB version : 1.5.2


## 4. Création des dimensions

In [4]:
# --- dim_usine ---
dim_usine = df[["usine", "pays"]].drop_duplicates().reset_index(drop=True)
dim_usine["usine_key"] = range(1, len(dim_usine) + 1)
con.register("dim_usine_df", dim_usine)
con.execute("CREATE TABLE dim_usine AS SELECT * FROM dim_usine_df")
print(f"dim_usine : {len(dim_usine)} lignes")
dim_usine

dim_usine : 2 lignes


,usine,pays,usine_key
0,ES-01,Espagne,1
1,FR-01,France,2


In [5]:
# --- dim_produit ---
dim_produit = pd.DataFrame({
    "Type": ["L", "M", "H"],
    "gamme": ["Low (grand public)", "Medium (semi-premium)", "High (haute precision)"],
    "produit_key": [1, 2, 3],
})
con.register("dim_produit_df", dim_produit)
con.execute("CREATE TABLE dim_produit AS SELECT * FROM dim_produit_df")
print(f"dim_produit : {len(dim_produit)} lignes")
dim_produit

dim_produit : 3 lignes


,Type,gamme,produit_key
0,L,Low (grand public),1
1,M,Medium (semi-premium),2
2,H,High (haute precision),3


In [6]:
# --- dim_temps ---
ts_min = pd.to_datetime(df["timestamp"]).min().floor("D")
ts_max = pd.to_datetime(df["timestamp"]).max().ceil("D")
dates = pd.date_range(ts_min, ts_max, freq="D")

dim_temps = pd.DataFrame({
    "jour": dates.date,
    "annee": dates.year,
    "mois": dates.month,
    "semaine": dates.isocalendar().week.values,
    "jour_semaine": dates.day_name(),
})
dim_temps["temps_key"] = range(1, len(dim_temps) + 1)
con.register("dim_temps_df", dim_temps)
con.execute("CREATE TABLE dim_temps AS SELECT * FROM dim_temps_df")
print(f"dim_temps : {len(dim_temps)} jours")
dim_temps.head()

dim_temps : 32 jours


,jour,annee,mois,semaine,jour_semaine,temps_key
0,2025-10-01,2025,10,40,Wednesday,1
1,2025-10-02,2025,10,40,Thursday,2
2,2025-10-03,2025,10,40,Friday,3
3,2025-10-04,2025,10,40,Saturday,4
4,2025-10-05,2025,10,40,Sunday,5


In [7]:
# --- dim_machine ---
dim_machine = dim_usine.copy()
dim_machine["machine_id"] = "CNC-" + dim_machine["usine"]
dim_machine["machine_type"] = "Fraiseuse CNC 5 axes"
dim_machine["machine_key"] = range(1, len(dim_machine) + 1)
dim_machine = dim_machine[["machine_key", "machine_id", "machine_type", "usine"]]
con.register("dim_machine_df", dim_machine)
con.execute("CREATE TABLE dim_machine AS SELECT * FROM dim_machine_df")
print(f"dim_machine : {len(dim_machine)} machines")
dim_machine

dim_machine : 2 machines


,machine_key,machine_id,machine_type,usine
0,1,CNC-ES-01,Fraiseuse CNC 5 axes,ES-01
1,2,CNC-FR-01,Fraiseuse CNC 5 axes,FR-01


## 5. Création de la table de faits

In [8]:
fact = df.merge(dim_usine[["usine", "usine_key"]], on="usine")
fact = fact.merge(dim_produit[["Type", "produit_key"]], on="Type")
fact = fact.merge(dim_machine[["usine", "machine_key"]], on="usine")
fact["jour"] = pd.to_datetime(fact["timestamp"]).dt.date
fact = fact.merge(dim_temps[["jour", "temps_key"]], on="jour")

fact_production = fact.rename(columns={
    "UDI": "piece_id",
    "Air temperature [K]": "air_temp_k",
    "Process temperature [K]": "process_temp_k",
    "Rotational speed [rpm]": "rotational_speed_rpm",
    "Torque [Nm]": "torque_nm",
    "Tool wear [min]": "tool_wear_min",
    "Machine failure": "machine_failure",
})[[
    "piece_id", "timestamp", "usine_key", "produit_key", "machine_key", "temps_key",
    "air_temp_k", "process_temp_k", "rotational_speed_rpm", "torque_nm", "tool_wear_min",
    "machine_failure", "TWF", "HDF", "PWF", "OSF", "RNF"
]]

con.register("fact_df", fact_production)
con.execute("CREATE TABLE fact_production AS SELECT * FROM fact_df")
print(f"fact_production : {len(fact_production)} lignes")
fact_production.head()

fact_production : 10070 lignes


,piece_id,timestamp,usine_key,produit_key,machine_key,temps_key,air_temp_k,process_temp_k,rotational_speed_rpm,torque_nm,tool_wear_min,machine_failure,TWF,HDF,PWF,OSF,RNF
0,319,2025-10-01 06:18:00.648612845,1,1,1,1,297.8,308.4,1512,39.596343,189,0,0,0,0,0,0
1,1150,2025-10-01 06:24:32.591039184,1,1,1,1,297.2,308.1,1446,38.351136,118,0,0,0,0,0,0
2,971,2025-10-01 06:27:44.851190073,1,1,1,1,295.8,306.5,1568,35.457069,119,0,0,0,0,0,0
3,6659,2025-10-01 06:41:16.224233662,1,1,1,1,301.4,310.4,1430,49.233544,186,0,0,0,0,0,0
4,3458,2025-10-01 06:42:46.064257193,1,2,1,1,301.5,310.4,2188,10.963999,29,1,0,0,1,0,0


## 6. Tables d'agrégats précalculées

In [9]:
# On reprend les KPIs du notebook 04 s'ils sont deja exportes, sinon on recalcule
kpis_jour_path = GOLD_DIR / "kpis_jour.parquet"
kpis_heure_path = GOLD_DIR / "kpis_heure.parquet"

if kpis_jour_path.exists():
    kpis_jour = pd.read_parquet(kpis_jour_path)
    con.register("kj_df", kpis_jour)
    con.execute("CREATE TABLE fact_kpis_jour AS SELECT * FROM kj_df")
    print(f"fact_kpis_jour : {len(kpis_jour)} lignes")

if kpis_heure_path.exists():
    kpis_heure = pd.read_parquet(kpis_heure_path)
    con.register("kh_df", kpis_heure)
    con.execute("CREATE TABLE fact_kpis_heure AS SELECT * FROM kh_df")
    print(f"fact_kpis_heure : {len(kpis_heure)} lignes")

fact_kpis_jour : 62 lignes
fact_kpis_heure : 1439 lignes


## 7. Vues consolidées pour le dashboard

In [10]:
con.execute('''
CREATE VIEW v_production AS
SELECT
    f.piece_id,
    f.timestamp,
    u.usine,
    u.pays,
    p.gamme,
    m.machine_id,
    f.torque_nm,
    f.rotational_speed_rpm,
    f.tool_wear_min,
    f.process_temp_k,
    f.air_temp_k,
    f.machine_failure,
    f.TWF, f.HDF, f.PWF, f.OSF, f.RNF
FROM fact_production f
JOIN dim_usine u   ON f.usine_key = u.usine_key
JOIN dim_produit p ON f.produit_key = p.produit_key
JOIN dim_machine m ON f.machine_key = m.machine_key
''')
print("Vue v_production creee")

Vue v_production creee


## 8. Validation avec requêtes exemple

In [11]:
# Top 5 des jours les plus problematiques
print("Top 5 jours avec le plus de pannes :")
print(con.execute('''
SELECT
    u.usine,
    t.jour,
    COUNT(*) AS pieces,
    SUM(f.machine_failure) AS pannes,
    ROUND(100.0 * SUM(f.machine_failure) / COUNT(*), 2) AS pct_rebut
FROM fact_production f
JOIN dim_usine u ON f.usine_key = u.usine_key
JOIN dim_temps t ON f.temps_key = t.temps_key
GROUP BY u.usine, t.jour
ORDER BY pct_rebut DESC
LIMIT 5
''').df())

Top 5 jours avec le plus de pannes :
   usine       jour  pieces  pannes  pct_rebut
0  ES-01 2025-10-03     133     8.0       6.02
1  FR-01 2025-10-29     188    11.0       5.85
2  FR-01 2025-10-09     207    12.0       5.80
3  FR-01 2025-10-04     213    12.0       5.63
4  FR-01 2025-10-31      56     3.0       5.36


In [12]:
# Rebut par gamme de piece
print("\nTaux de rebut par gamme :")
print(con.execute('''
SELECT
    p.gamme,
    COUNT(*) AS pieces,
    SUM(f.machine_failure) AS pannes,
    ROUND(100.0 * SUM(f.machine_failure) / COUNT(*), 2) AS pct_rebut
FROM fact_production f
JOIN dim_produit p ON f.produit_key = p.produit_key
GROUP BY p.gamme
ORDER BY pct_rebut DESC
''').df())


Taux de rebut par gamme :
                    gamme  pieces  pannes  pct_rebut
0      Low (grand public)    6041   233.0       3.86
1   Medium (semi-premium)    3019    79.0       2.62
2  High (haute precision)    1010    20.0       1.98


In [13]:
# Liste des tables et vues creees
print("\nTables dans l'entrepot :")
print(con.execute("SHOW TABLES").df())


Tables dans l'entrepot :
               name
0       dim_machine
1    dim_machine_df
2       dim_produit
3    dim_produit_df
4         dim_temps
5      dim_temps_df
6         dim_usine
7      dim_usine_df
8           fact_df
9   fact_kpis_heure
10   fact_kpis_jour
11  fact_production
12            kh_df
13            kj_df
14     v_production


## 9. Fermeture de la connexion

In [14]:
con.close()
print(f"\nEntrepot construit : {DB_PATH}")
print(f"Taille : {DB_PATH.stat().st_size / 1024:.1f} Ko")


Entrepot construit : /home/romaric420/MSPR/data/gold/mecha.duckdb
Taille : 2060.0 Ko


## 10. Synthèse

### Ce qu'on a produit

- Base DuckDB `mecha.duckdb` avec modèle étoile complet
- 4 dimensions + 1 table de faits + 2 tables d'agrégats + 1 vue consolidée
- Requêtes SQL de validation fonctionnelles

### Grille MSPR couverte

- **C7** : entrepôt multidimensionnel conçu, argumenté, déployé
- **C2** : architecture BI 3 couches (collecte notebook 02 / stockage notebook 05 / restitution dashboard)

### Suite

- **Notebook 06** : feature engineering pour augmenter le pouvoir prédictif du modèle ML
